# Setup

In [ ]:
# Colab-specific setup
import pathlib

if 'google.colab' not in str(get_ipython()):
    base_folder = pathlib.Path('../../')
else:
    # install extra notebook dependencies in Colab
    ! uv pip install 'watermark==2.4.*' 'pyyaml==6.0.*'

    # mount colab folder
    #from google.colab import drive
    #drive.mount('/content/drive')
    #base_folder = pathlib.Path('/content/drive/MyDrive/Vision/')

    # directly download the prepared dataset and shared helpers
    # (Colab only ever gets the 'small' set -- 'full' is local-only, see 1-preparation.ipynb)
    ! mkdir -p 'results'
    ! wget -nv -P 'prepared' https://raw.githubusercontent.com/mgmalheiros/vision/master/prepared/1-coins-small-image.zip
    ! wget -nv -P 'prepared' https://raw.githubusercontent.com/mgmalheiros/vision/master/prepared/1-coins-small-labels.csv
    ! wget -nv https://raw.githubusercontent.com/mgmalheiros/vision/master/process/counting/common.py
    base_folder = pathlib.Path('./')

In [ ]:
# show library versions
import watermark

# scikit-image also installs imageio and pillow
print(watermark.watermark(packages='skimage,imageio,PIL,pandas,scipy,yaml'))

# Loading

In [ ]:
# 'small' = the 100-image demo set tracked in the repo. 'full' = the 6021-image
# dataset prepared locally by 1-preparation.ipynb -- see the note there. Only
# 'small' exists on Colab, so leave this as 'small' unless running locally.
dataset_size = 'full'  # 'small' or 'full'

import common

images = common.load_prepared_images(base_folder / 'prepared' / f'1-coins-{dataset_size}-image.zip')
df = common.load_labels(base_folder / 'prepared' / f'1-coins-{dataset_size}-labels.csv')

print(f'{len(images)} images loaded')
df.describe()

# Edge-based Method
Find coin boundaries with Canny edge detection, close them into solid blobs with a
dilate &rarr; fill-holes &rarr; erode sequence, then count the resulting regions
(filtering out anything smaller than `min_area`, which discards noise left over
from the morphological cleanup).

In [ ]:
import numpy as np
from scipy import ndimage as ndi
from skimage import morphology
from skimage.color import label2rgb
from skimage.feature import canny
from skimage.measure import regionprops

In [ ]:
def detect_edges(image, sigma=4.3, dilation_disk=7, erosion_disk=29, min_area=40):
    edges = canny(image, sigma=sigma)
    dilated = morphology.dilation(edges, morphology.disk(dilation_disk))
    filled = ndi.binary_fill_holes(dilated)
    eroded = morphology.erosion(filled, morphology.disk(erosion_disk))

    labeled, _ = ndi.label(eroded)
    props = regionprops(labeled)
    count = sum(1 for r in props if r.area >= min_area)
    return count, labeled

## Visual check

In [ ]:
for name in sorted(images)[:3]:
    image = images[name]
    gt = common.real_count(df, name)

    edges = canny(image, sigma=4.3)
    common.P(image, 'original', size=6, cmap='gray')
    common.P(edges, 'canny', size=6, cmap='gray')

    count, labeled = detect_edges(image)
    result = label2rgb(labeled, image=image)
    common.P(result, f'Real: {gt} | Found: {count}', size=6)
    common.S()

# Evaluation
Run the detector over the whole prepared dataset, score it against `real_count`, and
write a summary (accuracy, mean absolute error, time, memory) to `results/edges_results.yaml`
(or `edges_results-full.yaml` when `dataset_size == 'full'`, so a full run never overwrites
the small-dataset results).

In [ ]:
results, summary = common.evaluate_method(
    images, df,
    lambda image: detect_edges(image)[0],
    method_name='Canny edges',
    parameters={'sigma': 4.3, 'dilation_disk': 7, 'erosion_disk': 29, 'min_area': 40},
    results_path=base_folder / 'results' / f'edges_results{"" if dataset_size == "small" else "-full"}.yaml',
)

for key, value in summary.items():
    print(f'{key}: {value}')

In [ ]:
results[['abs_error', 'time_seconds', 'peak_memory_kb']].describe()